In [1]:
# Training Arabic Sentiment Model Notebook
import pandas as pd
import joblib
from Preprocessing_pipeline import normalize_arabic
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

df = pd.read_csv('../data/arabic_reviews.csv')
df = df.dropna(subset=['review_description', 'rating']).copy()
df['clean_text'] = df['review_description'].apply(normalize_arabic)

def map_sentiment(val):
    val = str(val).strip().lower()
    if val in ['positive', '5', '4', 'ممتاز', 'إيجابي']:
        return 'Positive'
    elif val in ['negative', '1', '2', 'سيء', 'سلبي']:
        return 'Negative'
    return 'Neutral'

df['sentiment'] = df['rating'].apply(map_sentiment)

X_train, X_test, y_train, y_test = train_test_split(df['clean_text'], df['sentiment'], test_size=0.2, random_state=42)

model = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), max_features=15000)),
    ('clf', LogisticRegression(C=2.0, max_iter=1000))
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

joblib.dump(model, 'Arabic_model_weights.pkl')
print("Model saved to Arabic_model_weights.pkl")


              precision    recall  f1-score   support

    Negative       0.83      0.80      0.81      2832
     Neutral       0.28      0.04      0.07       405
    Positive       0.85      0.93      0.89      4772

    accuracy                           0.84      8009
   macro avg       0.65      0.59      0.59      8009
weighted avg       0.81      0.84      0.82      8009

Model saved to Arabic_model_weights.pkl
